# Notebook 2: nested hyperparameter selection, and what it cannot buy youCompanion to `REM_Turku_handoff.ipynb`. Read that one first.**The short version, so you can decide whether to spend GPU here at all.** We ran a12-config nested sweep across two architectures and three targets on REM_Turku, and amatching one on 101-Nights body_action. **Nesting the selection changed nothing**, onevery arm, and the reason is measurable rather than anecdotal: at these sample sizes theinner-validation scores of competing configs are separated by less than their own noise.Defaults are not a shortcut here. They are the honest answer.

## 1. The evidence, three independent measurements**A 14-config sweep reproduced library defaults to 0.0006.**apprehension: defaults 0.6462, swept 0.6456. anger: 0.4769 vs 0.4730.**The dev split could not rank the configs it was selecting from.** All 14 scored between0.426 and 0.515 on the 6-subject dev fold, inside the binomial SE at that fold size.Taking the argmax of draws that are all at chance selects noise, and we reported that as afinding rather than as tuning.**Nested selection inside every outer fold changed nothing across five arms.**| arch / target | nested | frozen config | delta ||---|---|---|---|| shallow_bd apprehension | 0.5571 | 0.5413 | +0.016 || shallow_bd confusion | 0.4911 | 0.4961 | -0.005 || eegnet apprehension | 0.5303 | 0.5507 | -0.020 || eegnet anger | 0.4823 | ~0.475 | +0.007 || eegnet confusion | 0.5419 | ~0.496 | +0.046 |Every delta is inside the fold-to-fold sd (0.08 to 0.16). We pre-registered the predictionthat nesting would LOWER the numbers by removing optimism, and were wrong five times:there was no optimism to remove.**The inner spans are the actual finding.** Across those arms the 12 sampled configs span**0.388 to 0.631** on inner validation, and 0.10 to 0.20 within any single arm. Onbody_action the gap is starker still: inner-validation winners at 0.64-0.65 delivering0.38-0.46 on the held-out fold, below the 0.5385 majority baseline.**The reportable sentence is not "tuning does not help here". It is: at 113 awakenings theconfig-selection signal is below the noise floor, measured.** That is a claim about thedata regime and it transfers to anyone working at this scale.

## 2. Why nesting matters even when it changes nothingA sweep that selects on one dev split and then reports accuracy computed from that samesplit family produces a number whose optimism you cannot bound. Nesting removes thequestion. Ours came back unchanged, which is a *result about the data*, not a licence toskip the control.Per outer fold: sample 12 configs from the method's own grid, score each on 3 inner splitsof that fold's **training** subjects, retrain the winner on all of them, score once on theheld-out subject. The held-out subject never touches selection, and no config list isshared between folds.

In [ ]:
import numpy as np, itertoolsdef sample_configs(space, n, rng):    """n configs from a dict of {key: [values]}. Log-uniform for lr-like keys."""    out = []    for _ in range(n):        cfg = {}        for k, v in space.items():            if k in ("lr", "max_lr"):                cfg[k] = float(10 ** rng.uniform(np.log10(min(v)), np.log10(max(v))))            else:                cfg[k] = v[int(rng.integers(len(v)))]        out.append(cfg)    return out# Ninon's constraints, and they are not optional:#   F1 * D <= n_channels, NOT <= 64. The repo's validate_config hardcoded 64, which on a#   24-channel montage silently admits configs whose depthwise stage is wider than the#   input.#   depthwise_kernel_length must genuinely vary; it had been pinned to [256] while the#   constructor default is 32.def grid_for(n_channels):    return {        "F1_D": [(f, d) for f in (4, 8, 16) for d in (2, 4) if f*d <= n_channels],        "kernel_length": [32, 64, 96, 128, 160],        "depthwise_kernel_length": [16, 32, 64, 128, 256],        "separable_kernel_length": [8, 16, 32],        "drop_prob": [0.3, 0.4, 0.5, 0.6, 0.7],        "lr": [1e-4, 1e-2], "batch_size": [32, 64], "epochs": [60, 100],        "optimizer": ["adam", "adamw"],    }print("grid for 24 channels:", {k: len(v) for k, v in grid_for(24).items()})

In [ ]:
def nested_evaluate(make_fit_predict, target, budget=3, inner_folds=3, seed=0, verbose=True):    """Nested selection. DEFAULT budget=3 and one target so this finishes on a Colab GPU.    Our runs used budget=12, inner_folds=3, all three targets: roughly 37 trainings per    outer fold, 8-12 hours per arm on an A100-class card. Set budget=12 only if you have    that time; the sections above are why we do not think it will change your answer.    make_fit_predict(cfg) -> fit_predict, so the config actually reaches your model.    """    X, y, s = load(target)    space = grid_for(X[0].shape[1])    per_subject, chosen = {}, {}    for held in sorted(set(s), key=int):        te = s == held; tr = ~te        if y[te].sum() in (0, te.sum()):            per_subject[held] = None; continue        tr_subs = [u for u in sorted(set(s), key=int) if u != held]        cfgs = sample_configs(space, budget, np.random.default_rng(1000 + int(held) + seed))        rng = np.random.default_rng(2000 + int(held) + seed)        inner = np.array_split(rng.permutation(tr_subs), inner_folds)        scores = []        for cfg in cfgs:            vals = []            for f in inner:                iv = np.isin(s, list(f)) & tr; it = tr & ~iv                if y[iv].sum() in (0, iv.sum()) or len(set(y[it])) < 2: continue                win = lambda m: (np.concatenate([X[i][::2] for i in np.where(m)[0]]) * 1e6,                                 np.concatenate([[y[i]]*len(X[i][::2]) for i in np.where(m)[0]]))                Xa, ya = win(it); Xb, yb = win(iv)                vals.append(bal_acc(np.asarray(make_fit_predict(cfg)(Xa, ya, Xb)), yb))            # a config that failed EVERY inner fold gets NaN, never 0.0: scoring it zero            # ties it with every other failure and argmax silently returns config 0            scores.append(float(np.mean(vals)) if vals else float("nan"))        if all(x != x for x in scores):            raise SystemExit("every config failed on every inner fold; fix the plumbing "                             "before trusting any number")        best = int(np.nanargmax(scores))        win = lambda m: (np.concatenate([X[i][::2] for i in np.where(m)[0]]) * 1e6,                         np.concatenate([[y[i]]*len(X[i][::2]) for i in np.where(m)[0]]))        Xa, ya = win(tr); Xb, yb = win(te)        per_subject[held] = bal_acc(np.asarray(make_fit_predict(cfgs[best])(Xa, ya, Xb)), yb)        chosen[held] = {"inner": scores[best], "spread": [float(np.nanmin(scores)),                                                          float(np.nanmax(scores))]}        if verbose:            print(f"  subj {held}: inner {scores[best]:.4f} "                  f"(spread {np.nanmin(scores):.3f}-{np.nanmax(scores):.3f}) "                  f"-> OUTER {per_subject[held]:.4f}")    v = [x for x in per_subject.values() if x is not None]    if verbose: print(f"nested {target}: {np.mean(v):.4f} over {len(v)} subjects")    return {"per_subject": per_subject, "chosen": chosen, "mean": float(np.mean(v))}print("nested_evaluate ready; default budget=3 for Colab, our runs used 12")

## 3. Read the spread, not just the winnerThe most useful output above is not `mean`. It is `chosen[subject]["spread"]`. If your 12configs span less on inner validation than the fold-to-fold sd of your outer scores, theselection is choosing noise and the winner carries no information. Print it every time:```pythonr = nested_evaluate(my_factory, "apprehension", budget=12)spans = [c["spread"][1] - c["spread"][0] for c in r["chosen"].values()]print("inner span per fold:", np.round(spans, 3))```Ours ran 0.10 to 0.20 per fold against an outer sd of 0.08 to 0.16. That is the diagnosticthat told us tuning could not help, and it costs nothing to compute.